In [20]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# 1. Load 
df_av = pd.read_csv("av_timeseries.csv")
df_rss = pd.read_csv("rss_news_sentiment.csv")
df_yfinance = pd.read_csv("yahoo_finance_historical.csv", skiprows=2)
df_yfinance.columns = ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Price_Movement', 'Direction', 'Volatility']

# 2. Standardize
df_yfinance['Date'] = pd.to_datetime(df_yfinance['Date']).dt.tz_localize(None)
df_av['Date'] = pd.to_datetime(df_av['Date']).dt.tz_localize(None)
df_rss['Date'] = pd.to_datetime(df_rss['Date']).dt.normalize().dt.tz_localize(None)

# 3. Sentiment Mapping
sentiment_map = {'Positive': 1, 'Neutral': 0, 'Negative': -1}
df_rss['Sentiment_Score'] = df_rss['Sentiment'].map(sentiment_map)
df_rss_daily = df_rss.groupby('Date')['Sentiment_Score'].mean().reset_index()

# 4. The "Safety" Merge (Use 'left' to keep price data even if news is missing)
# We keep df_yfinance as our "anchor"
final_df = pd.merge(df_yfinance, df_av, on="Date", how="left", suffixes=('_yf', '_av'))
final_df = pd.merge(final_df, df_rss_daily, on="Date", how="left")

# 5. Fill Missing Data (If news is missing for a day, we assume 'Neutral' or 0)
final_df['Sentiment_Score'] = final_df['Sentiment_Score'].fillna(0)
final_df = final_df.ffill().bfill() # Fill remaining price gaps

# 6. Final Sort
final_df = final_df.sort_values('Date').reset_index(drop=True)

print(f" Success! Master Dataset Created.")
print(f"Total Rows: {len(final_df)}") 
print(f"Date Range: {final_df['Date'].min().date()} to {final_df['Date'].max().date()}")

final_df.head()

 Success! Master Dataset Created.
Total Rows: 1590
Date Range: 2020-01-02 to 2026-04-30


,Date,Close_yf,High_yf,Low_yf,Open_yf,Volume_yf,Price_Movement_yf,Direction_yf,Volatility_yf,Open_av,High_av,Low_av,Close_av,Volume_av,Price_Movement_av,Direction_av,Volatility_av,Sentiment_Score
0,2020-01-02,3257.850098,3258.139893,3235.530029,3244.669922,3459930000,13.180176,Up,22.609863,686.59,686.64,681.57,683.63,53033654.0,-2.96,Down,5.07,0.0
1,2020-01-03,3234.850098,3246.149902,3222.340088,3226.360107,3484700000,8.489990,Up,23.809814,686.59,686.64,681.57,683.63,53033654.0,-2.96,Down,5.07,0.0
2,2020-01-06,3246.280029,3246.840088,3214.639893,3217.550049,3702460000,28.729980,Up,32.200195,686.59,686.64,681.57,683.63,53033654.0,-2.96,Down,5.07,0.0
3,2020-01-07,3237.179932,3244.909912,3232.429932,3241.860107,3435910000,-4.680176,Down,12.479980,686.59,686.64,681.57,683.63,53033654.0,-2.96,Down,5.07,0.0
4,2020-01-08,3253.050049,3267.070068,3236.669922,3238.590088,3726840000,14.459961,Up,30.400146,686.59,686.64,681.57,683.63,53033654.0,-2.96,Down,5.07,0.0
